## Stanford CS 329a Self-Improving AI Agents, Homework 1

In this homework, we will go over basic techniques to scale test-time compute and self-improve. 

The homework will have 5 parts:

1. Zero-shot Evaluation (10 points)
2. Majority Voting (30 points)
3. Best-of-N with a Generative Reward Model (30 points)
4. Self-Improvement with Feedback (45 points)
5. Analysis (15 points)





In [ ]:
%load_ext autoreload
%autoreload 2

## Importing the relevant tools.
## Make sure you installed this package with the instructions in README.md
import os
import numpy as npcan
import matplotlib.pyplot as plt
from dotenv import load_dotenv

from cs329_hw1.tasks import AIME25
from cs329_hw1.methods import get_sampler, get_verifier

# Load environment variables from .env file
load_dotenv()

# Configuration
HF_TOKEN = os.getenv("HUGGINGFACE_HUB_TOKEN")
DEBUG_MODE = False  # Set to True when developing, False for final results

###  Dataset: AIME25
We will work with the AIME25 dataset from [Hugging Face](https://huggingface.co/datasets/math-ai/aime25).
This dataset contains 30 challenging mathematical problems from the American Invitational Mathematics Examination (AIME).
These problems are significantly more difficult than typical high school math problems and require advanced problem-solving techniques.
<br>
First, let's get familiar with the dataset.

In [ ]:
aime25 = AIME25(hf_token=HF_TOKEN)
problems = aime25.get_problems(debug_mode=DEBUG_MODE)
system_prompt = aime25.get_system_prompt()

# Each problem is a dictionary with a "problem", "answer", and "id"
print(f"Loaded {len(problems)} AIME25 problems")
print("Problem: ", problems[0]["problem"])
print("answer: ", problems[0]["answer"])

Here, each problem is a dictionary with a "problem" and an "answer". 
<br>
"problem" is the statement of the mathematical question, and "answer" is the correct answer to the question. Importantly, this is not a multiple-choice question, so the answer can often be a mathematical expression.


###  Useful tools
To help you implement the techniques we will cover in this homework, we implemented a few useful methods for you under `cs329_hw1.methods`. We recommend you look through this function, and later fucntions we provide, to understand the capabilities of how they can be used for the homework!

In [ ]:
# We implement a few useful methods for you under `methods`. 

# First, the simple multiple sampler.
# This method returns a list of lists, where for each prompt that is passed as an input, we return a list of responses.
# You will use this method as the basis of the techniques we will implement.
# See the below example where we sample 3 responses from together_ai/Qwen/Qwen3-Next-80B-A3B-Instruct with a temperature of 0.7.

method = get_sampler("sample_multiple", "together_ai/Qwen/Qwen3-Next-80B-A3B-Instruct", temperature=0.7, system_prompt=system_prompt)
prompts = ["What language model are you based on?"]
responses = method(prompts)
print("\n".join(responses[0]))


### Testing on a single problem

In [ ]:
# You can use this method function to get predictions for a given problem.
# We get our first prediction from the method.
# Let's get three predictions for the first problem, and read them out.
problem_prompts = [problems[0]["problem"]]
predictions = method(problem_prompts)[0]
print("Sample response: ".join(predictions))


### Verifier

In [ ]:
# We give you a verifier tool for the AIME25 dataset.
# The verifier for AIME25 is a simple tool that checks if the prediction is correct.
# Internally, it parses the prediction and the answer and compares the final mathematical expression.
# Importantly, the verifier is not perfect due to parsing challenges. 
# We mostly re-use the Qwen verifier based on the [Qwen-2.5 MATH repository](https://github.com/QwenLM/Qwen2.5-Math/tree/main/evaluation).
# If you can be nerdsniped into writing a better verifier, we'd love to see it!

# Let's test the predictions we got for the first problem that we have.
verifier = get_verifier("aime25")

print("Correct answer: ", problems[0]["answer"])
for prediction in predictions:
    print("Last line of the prediction: ", prediction.split("\n\n")[-1])
    print("Is correct: ", verifier.verify(prediction, problems[0]["answer"]))

## Your turn!

### 1- Evaluating zero-shot predictions (10 points)

First, we will evaluate the accuracy of the predictions with a single sample, without using any test-time compute techniques. <br>
You will only need to use the `method` we defined above and the `verifier` to compute the accuracy.


Deliverable: 
- Write your code in the section specified by `TODO: YOUR CODE STARTS HERE` and `TODO: YOUR CODE ENDS HERE`.
- Report the accuracy of the predictions below.
- A zero_shot_correctness of type List[int] that will be used later to compare results

In [ ]:
method = get_sampler("sample_multiple", "together_ai/Qwen/Qwen3-Next-80B-A3B-Instruct", temperature=0.7, n_samples=1, system_prompt=system_prompt)

test_problems = problems
zero_shot_correctness = []

### TODO: YOUR CODE STARTS HERE


### TODO: YOUR CODE ENDS HERE
### Report the accuracy of the predictions below.
print(f"Accuracy: {sum(zero_shot_correctness) / len(zero_shot_correctness)}")

### 2- Majority Voting (30 points)

Here we will implement our first test-time compute technique, majority voting, as described in [this paper](https://arxiv.org/abs/2408.03314) or [this earlier paper](https://arxiv.org/abs/2203.11171). In particular,
- You will sample multiple (in this case, 16) responses for each problem.
- You will then take the majority vote as the prediction. The voting will be performed per each normalized expression (i.e., given the entire solution, we will parse the final numerical expression and perform the voting on that). We provide utility functions to do this.
- You will then evaluate the prediction against the ground truth.

Deliverable: 
- Write your code in the section specified by `TODO: YOUR CODE STARTS HERE` and `TODO: YOUR CODE ENDS HERE`.
- Report the accuracy of the predictions below.

In [ ]:
## 2.1 - Implementing majority voting.
from typing import List, Union
from cs329_hw1.methods.simple_samplers import SampleMultiple
from cs329_hw1.tasks.math_utils import (
    strip_string,
    extract_answer,
)

class MajorityVoting:
    """
    A class that implements majority voting strategy using multiple samples.
    It generates multiple responses for each prompt and selects the most common answer.
    """

    def __init__(
        self,
        model: str,
        system_prompt: str = None,
        n_samples: int = 5,
        temperature: float = 0.7,
        max_workers: int = 16
    ):
        """
        Initialize the majority voting method.

        Args:
            model (str): The name of the model to use
            system_prompt (str, optional): System prompt to use for the model
            n_samples (int, optional): Number of samples to generate per prompt. Defaults to 5.
            temperature (float, optional): Temperature for sampling. Defaults to 0.7.
        """
        self.sampler = SampleMultiple(
            model=model,
            system_prompt=system_prompt,
            n_samples=n_samples,
            temperature=temperature,
            max_workers=max_workers
        )

    def _parse_answer(self, response: str) -> str:
        return strip_string(extract_answer(response, "math"))

    def _get_majority_answer(self, responses: List[str]) -> str:
        """
        Determine the majority answer from a list of responses using a simple counter.

        Args:
            responses (List[str]): List of model responses

        Returns:
            str: The most common answer
        """
        assert isinstance(responses, list), "Responses must be a list"
        assert all(
            isinstance(r, str) for r in responses
        ), "All responses must be strings"
        # Extract answers from responses

        ## TODO: YOUR CODE STARTS HERE
        ## Implement the majority voting logic here.
        ## Feel free to use the `_parse_answer` method we implemented for you.
        # Create a counter for each unique answer
        
        ## TODO: YOUR CODE ENDS HERE
        return majority_answer

    def __call__(self, prompts: Union[str, List[str]], majority_voting_levels: List[int] = [1, 2, 4, 8, 16]) -> List[str]:
        """
        Execute majority voting on given prompt(s).

        Args:
            prompts (str or List[str]): The input prompt(s) to process

        Returns:
            List[List[str]]: For each majority voting level, we return the majority answer.
        """
        
        majority_answers = []
        responses = self.sampler(prompts)
        for majority_voting_level in majority_voting_levels:
            majority_answers.append([self._get_majority_answer(responses[i][:majority_voting_level]) for i in range(len(responses))])
        return majority_answers

In [ ]:
method = MajorityVoting(
    model="together_ai/Qwen/Qwen3-Next-80B-A3B-Instruct",
    n_samples=16,
    temperature=0.7,
    system_prompt=system_prompt,
    max_workers=32
)

In [ ]:
## Do not modify the code below; it is used to evaluate the accuracy of the predictions across different majority voting budgets.

test_problems = problems

total = len(test_problems)
prompts = [problem["problem"] for problem in test_problems]

majority_voting_levels = [1, 2, 4, 8, 16]
# Get all responses at once (this uses internal threading)
all_responses_list = method(prompts)


majority_voting_accuracies = []
for responses_list, majority_voting_level in zip(all_responses_list, majority_voting_levels):
    correct = 0
    
    for prediction, problem in zip(responses_list, test_problems):
        # We do not re-normalize the prediction here, as we already normalized it in the majority voting step.
        # To do so, we set the normalize_prediction flag to False.
        if verifier.verify(prediction, problem["answer"], normalize_prediction=False):
            correct += 1
        
    accuracy = correct / len(test_problems)
    majority_voting_accuracies.append(accuracy)


In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter([1], [sum(zero_shot_correctness) / len(zero_shot_correctness)], s=100, color="darkorange", marker="o", label='No test-time compute')
plt.plot(majority_voting_levels, majority_voting_accuracies, marker='o', color="green", label='Majority Voting')
plt.xlabel('Test-time Budget (# of API calls)')
plt.ylabel('Accuracy')
plt.title('Accuracy vs Number of Samples in Majority Voting')
plt.legend()
plt.grid(True)
plt.show()

### 3- Best-of-N with a Generative Reward Model (30 points)

Here we will implement our second test-time compute technique, best-of-N with a generative reward model. In particular,
- You will sample multiple (in this case, 16) responses for each problem.
- You will then use an LLM to aggregate these responses and select the best answer. This is akin to best-of-N with a reward model as in [the paper](https://arxiv.org/abs/2408.03314), but we use an LLM as a judge instead of a discriminative reward model. This is sometimes referred to [Generative Verifiers](https://arxiv.org/abs/2408.15240) or [Generative Reward Models](https://arxiv.org/abs/2410.12832).
- You will then verify the prediction against the ground truth.
- (Bonus, 2 points) We give a default prompt for the generative reward model. If you can improve the existing prompt and improve the accuracy of the technique compared to the numbers you get otherwise, you will get 2 points of extra credit. There is not an absolute number to beat, but ideally you observe +5% improvement over the default prompt.

Deliverables:
- Write your code in the section specified by `TODO: YOUR CODE STARTS HERE` and `TODO: YOUR CODE ENDS HERE`.
- Report the figure below.

In [ ]:
class LLMVoting:
    """
    A class that uses an LLM to aggregate multiple responses and select the best answer.
    It generates multiple responses for each prompt and uses another LLM call to choose the best answer.
    """

    def __init__(
        self,
        model: str,
        system_prompt: str = None,
        n_samples: int = 5,
        temperature: float = 0.7,
        max_workers: int = 16
    ):
        """
        Initialize the LLM voting method.

        Args:
            model (str): The name of the model to use
            system_prompt (str, optional): System prompt to use for the model
            n_samples (int, optional): Number of samples to generate per prompt. Defaults to 5.
            temperature (float, optional): Temperature for sampling. Defaults to 0.7.
        """
        self.sampler = SampleMultiple(
            model=model,
            system_prompt=system_prompt,
            n_samples=n_samples,
            temperature=temperature,
            max_workers=max_workers
        )
        
        self.aggregator = SampleMultiple(
            model=model,
            system_prompt=system_prompt + """\n\nYou are a mathematical expert. You will be shown multiple solutions to a math problem.
Your task is to analyze these solutions and select the most likely correct answer.""",
            n_samples=1,
            temperature=0,  
            max_workers=max_workers
        )

    def _create_aggregation_prompts(self, problems: List[str], all_responses: List[List[str]]) -> List[str]:
        """
        Create prompts for aggregation in parallel.
        
        Args:
            problems (List[str]): List of original problems
            all_responses (List[List[str]]): List of response lists for each problem
            
        Returns:
            List[str]: List of prompts for the aggregator
        """
        return [
            f"""Here is a math problem:
{problem}

I have received multiple solutions. Here they are:

{chr(10).join(f'Solution {i+1}:{chr(10)}{r}' for i, r in enumerate(responses))}

Based on these solutions, restate the solution here that you think is most likely correct."""
            for problem, responses in zip(problems, all_responses)
        ]

    def __call__(self, prompts: Union[str, List[str]]) -> List[List[str]]:
        """
        Execute LLM-based voting on given prompt(s).

        Args:
            prompts (str or List[str]): The input prompt(s) to process

        Returns:
            List[List[str]]: The chosen answer(s) and all responses
        """
        # Get multiple samples for each prompt
        ## TODO: YOUR CODE STARTS HERE
        
        ## TODO: YOUR CODE ENDS HERE

In [ ]:
# Implement and test LLM Voting
llm_method = LLMVoting(
    model="together_ai/Qwen/Qwen3-Next-80B-A3B-Instruct",
    n_samples=16,
    temperature=0.7,
    system_prompt=system_prompt,
    max_workers=32
)

# Test LLM voting on the same problems
test_problems = problems
prompts = [problem["problem"] for problem in test_problems]

# Get LLM voting predictions
llm_predictions = llm_method(prompts)

# Calculate LLM voting accuracy
correct_llm_voting = 0
total_llm_voting = len(test_problems)

for prediction, problem in zip(llm_predictions, test_problems):
    if verifier.verify(prediction[0], problem["answer"]):
        correct_llm_voting += 1

llm_voting_accuracy = correct_llm_voting / total_llm_voting
print(f"LLM Voting Accuracy: {llm_voting_accuracy:.3f}")
print(f"Correct: {correct_llm_voting}/{total_llm_voting}")




### 4- Iterative Self-Improvement Using Execution Feedback

This section implements a self-improvement system inspired by the RLEF (Reinforcement Learning with Execution Feedback) approach described in [Gehring et al., 2024](https://arxiv.org/pdf/2410.02089). The system samples an initial solution, critiques it, and then regenerates an improved solution using the feedback.

**Key Innovation**: Instead of generating multiple independent samples, we use a single sample followed by targeted feedback and regeneration. This mimics how humans learn from mistakes and iteratively improve their solutions.

**Process Overview**:
1. **Initial Generation**: Generate one solution to the problem
2. **Critique Phase**: Analyze the solution for errors and provide constructive feedback
3. **Regeneration Phase**: Generate a new solution incorporating the feedback
4. **Evaluation**: Compare original vs. improved solution accuracy

**Requirements:**
- Implement a critique system using a single model for feedback generation
- Implement solution regeneration based on critiques
- Test on all 30 problems from the AIME25 dataset
- Compare accuracy before and after self-improvement
- Analyze the effectiveness of the feedback-based approach

In [ ]:
class SelfImprovementSystem:
    """
    A class that implements classic RLEF (Reinforcement Learning from Execution Feedback).
    Uses only Qwen model for both generation and critique.
    """

    def __init__(self, model: str, system_prompt: str = None, temperature: float = 0.7, max_workers: int = 8):
        # Main generator for initial solutions
        self.generator = SampleMultiple(
            model=model,
            system_prompt=system_prompt,
            n_samples=1,
            temperature=temperature,
            max_workers=max_workers
        )
        
        # Critic for providing feedback
        critic_system_prompt = """You are an expert mathematical critic. Your job is to analyze mathematical solutions and provide constructive feedback.

When given a problem and its solution, analyze it for:
1. Logical errors and reasoning gaps
2. Mathematical mistakes and calculation errors
3. Conceptual misunderstandings
4. Missing steps or incomplete reasoning

Provide specific, actionable feedback that can help improve the solution. If the solution is correct, acknowledge that but still suggest potential improvements or alternative approaches.

Format your feedback as:
- **Issues Found:** [List specific problems]
- **Suggestions:** [How to fix the issues]
- **Overall Assessment:** [Brief summary]"""

        self.critic = SampleMultiple(
            model=model,
            system_prompt=critic_system_prompt,
            n_samples=1,
            temperature=0.3,  # Lower temperature for more focused critiques
            max_workers=max_workers
        )
        
        # Regenerator for creating improved solutions based on feedback
        regenerator_system_prompt = """You are an expert mathematician who learns from feedback. When given a problem, your previous solution, and feedback about that solution, you should:

1. Carefully analyze the feedback provided
2. Identify the key issues that need to be addressed
3. Generate a new, improved solution that addresses the feedback
4. Ensure your new solution is logically sound and mathematically correct

Always acknowledge the feedback and explain how you've addressed the main issues in your new solution."""

        self.regenerator = SampleMultiple(
            model=model,
            system_prompt=regenerator_system_prompt,
            n_samples=1,
            temperature=temperature,
            max_workers=max_workers
        )

    def _generate_initial_solution(self, problem: str) -> str:
        """Generate the initial solution to a problem."""
        ## TODO: YOUR CODE STARTS HERE
        
        ## TODO: YOUR CODE ENDS HERE

    def _critique_solution(self, problem: str, solution: str) -> str:
        """
        Critique a solution and provide feedback.
        
        Args:
            problem (str): The original problem
            solution (str): The solution to critique
            
        Returns:
            str: Feedback about the solution
        """
        ## TODO: YOUR CODE STARTS HERE
        
        ## TODO: YOUR CODE ENDS HERE

    def _regenerate_solution(self, problem: str, original_solution: str, feedback: str) -> str:
        """
        Generate an improved solution based on feedback.
        
        Args:
            problem (str): The original problem
            original_solution (str): The original solution
            feedback (str): Feedback about the original solution
            
        Returns:
            str: The improved solution
        """
        ## TODO: YOUR CODE STARTS HERE
        
        ## TODO: YOUR CODE ENDS HERE

    def improve_solution(self, problem: str) -> dict:
        """
        Complete RLEF self-improvement process for a single problem.
        
        Args:
            problem (str): The problem to solve
            
        Returns:
            dict: Results including original solution, feedback, and improved solution
        """
        ## TODO: YOUR CODE STARTS HERE
        
        
        ## TODO: YOUR CODE ENDS HERE

        return {
            "problem": problem,
            "original_solution": original_solution,
            "feedback": feedback,
            "improved_solution": improved_solution
        }

    def evaluate_improvement(self, results: dict, correct_answer: str) -> dict:
        """
        Evaluate whether the improved solution is better than the original.
        
        Args:
            results (dict): Results from improve_solution
            correct_answer (str): The correct answer to the problem
            
        Returns:
            dict: Evaluation results including accuracy comparisons
        """
        ## TODO: YOUR CODE STARTS HERE
        
        ## TODO: YOUR CODE ENDS HERE
        return {
            "original_correct": original_correct,
            "improved_correct": improved_correct,
            "improvement": improvement, # bool
            "correct_answer": correct_answer
        } 

In [ ]:
# Initialize the Self-Improvement System (RLEF)
# This system uses only Qwen for generation, critique, and regeneration
self_improvement_system = SelfImprovementSystem(
    model="together_ai/Qwen/Qwen3-Next-80B-A3B-Instruct",
    temperature=0.7,
    system_prompt=system_prompt,
    max_workers=16
)

test_problems_size=len(problems)
test_problems = problems

print(f"Testing RLEF self-improvement system on {test_problems_size} problems...")
print("This will take several minutes as we generate initial solutions, critique them, and regenerate improved solutions.")
print("Using only Qwen for all steps (generation, critique, regeneration).")
print()

# Store results for analysis
improvement_results = []
original_accuracies = []
improved_accuracies = []

for i, problem in enumerate(test_problems):
    print(f"Processing problem {i+1}/{test_problems_size}...")
    
    # Run the complete RLEF self-improvement process
    result = self_improvement_system.improve_solution(problem["problem"])
    
    # Evaluate the improvement
    evaluation = self_improvement_system.evaluate_improvement(result, problem["answer"])
    
    # Store results
    improvement_results.append({
        "problem_index": i,
        "problem": problem["problem"],
        "correct_answer": problem["answer"],
        "result": result,
        "evaluation": evaluation
    })
    
    original_accuracies.append(evaluation["original_correct"])
    improved_accuracies.append(evaluation["improved_correct"])
    
    # Print progress
    if evaluation["improvement"]:
        print(f"  ✓ Improved from incorrect to correct!")
    elif evaluation["original_correct"] and evaluation["improved_correct"]:
        print(f"  → Both solutions correct")
    elif not evaluation["original_correct"] and not evaluation["improved_correct"]:
        print(f"  → Both solutions incorrect")
    else:
        print(f"  → Regressed from correct to incorrect")

print(f"\nCompleted processing {test_problems_size} problems!")

In [ ]:
# Analyze the results
print("=== RLEF Self-Improvement Analysis ===")
print()

# Calculate overall accuracies
original_accuracy = sum(original_accuracies) / len(original_accuracies)
improved_accuracy = sum(improved_accuracies) / len(improved_accuracies)
improvement_count = sum(1 for i in range(len(original_accuracies)) 
                       if improved_accuracies[i] and not original_accuracies[i])
regression_count = sum(1 for i in range(len(original_accuracies)) 
                     if original_accuracies[i] and not improved_accuracies[i])

print(f"Original Solution Accuracy: {original_accuracy:.3f} ({sum(original_accuracies)}/{len(original_accuracies)})")
print(f"Improved Solution Accuracy: {improved_accuracy:.3f} ({sum(improved_accuracies)}/{len(improved_accuracies)})")
print(f"Accuracy Improvement: {improved_accuracy - original_accuracy:.3f}")
print(f"Problems Improved: {improvement_count}")
print(f"Problems Regressed: {regression_count}")
print()

# Show detailed examples of improvements
print("=== Detailed Improvement Examples ===")
improvement_examples = [r for r in improvement_results if r["evaluation"]["improvement"]]

if improvement_examples:
    print(f"Found {len(improvement_examples)} successful improvements. Showing first example:")
    example = improvement_examples[0]
    
    print(f"\nProblem: {example['problem']}")
    print(f"Correct Answer: {example['correct_answer']}")
    print(f"\nOriginal Solution:")
    print(example['result']['original_solution'][:500] + "..." if len(example['result']['original_solution']) > 500 else example['result']['original_solution'])
    
    print(f"\nFeedback from Qwen Critic:")
    print(example['result']['feedback'][:500] + "..." if len(example['result']['feedback']) > 500 else example['result']['feedback'])
    
    print(f"\nImproved Solution:")
    print(example['result']['improved_solution'][:500] + "..." if len(example['result']['improved_solution']) > 500 else example['result']['improved_solution'])
else:
    print("No successful improvements found in this subset.")

## Analysis and Discussion (15 points)

### Deliverables

**Required**: Complete the following analysis tasks based on your experimental results from the previous sections.

### Task 1: Performance Comparison Graph (5 points)

Create a comprehensive comparison graph that includes:
1. Zero-shot baseline (single point)
2. Majority voting performance across sample sizes [1, 2, 4, 8, 16]
3. LLM voting performance (single point for 16 samples)
4. Self-improvement performance (single point showing improvement)

Requirements:
- Use the accuracy values you computed in previous sections
- Include proper labels, legend, and title
- Use different colors/markers for each method
- Set appropriate axis limits and grid
- Note: Self-improvement uses 3 API calls per problem (initial + critique + regeneration)

In [ ]:
# Task 1: Performance Comparison Graph
# TODO: YOUR CODE STARTS HERE

# TODO: YOUR CODE ENDS HERE

## **Task 2: Quantitative Analysis (5 points)**


Based on your experimental results, answer the following specific questions with numerical evidence:

1. **Accuracy Rankings**: List the methods in order of highest to lowest accuracy. Include the actual accuracy values.

**TODO** 


2. **Cost-Effectiveness**: Calculate the accuracy improvement per additional API call for majority voting (from n=1 to n=16) and compare it to self-improvement (3 API calls total). Which method provides better returns on additional compute?

**TODO**


3. **Self-Improvement Analysis**: 
   - What is the accuracy improvement from self-improvement compared to zero-shot?
   - Based on your error category analysis, which error type (logical, mathematical, conceptual, completeness) appears most frequently in incorrect solutions?
   - What percentage of problems showed successful improvement?

**TODO** 


## **Task 3: Method-Specific Insights (5 points)**


Address these specific questions about the methods you implemented:

1. **Majority Voting**: 
   - At what sample size does majority voting show diminishing returns? (Identify the point where accuracy improvement becomes minimal)
   - What is the maximum accuracy achieved by majority voting?

**TODO** 

2. **LLM Voting**: 
   - How does LLM voting accuracy compare to majority voting at n=16?
   - What is the computational cost difference between LLM voting and majority voting at n=16?

**TODO** 



3. **Self-Improvement**:
   - How does the self-improvement approach compare to traditional test-time compute methods?
   - What are the advantages and disadvantages of using feedback-based regeneration vs. multiple independent samples?
   - Based on your results, which error categories seem most important for guiding improvements?

**TODO** 


